# Does Anything Move The Score By 0.1?

One question, one chart: **is there a single condition, or a combination of
conditions, that confidently moves the score by at least 0.1?**

Everything below is computed at run time from the replayed comparisons and the
text sidecar. No figure in this notebook is written by hand.

Two measures appear, and they are not interchangeable:

- a **single condition** is measured by its **effect**: the mean score of the
  images carrying it minus the mean of the images not carrying it.
- a **combination** is measured by its **lift**: the mean score of the images
  matching every condition, minus the score predicted from those conditions
  acting independently. A raw gap for a combination would double-count the
  individual conditions and overstate what the combination itself contributes.

An interval alone is not enough to call anything a finding, and this notebook
does not stop at one. Three separate corrections apply, each for its own reason:
a family-wise threshold for screening thousands of conditions at once,
cross-validation for the fact that a search returns the best of many candidates
rather than a random one, and a permutation null for the bias that survives both
— which is what the search invents when it is pointed at noise.

In [ ]:
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

from comfyui_image_scorer.application.analysis.row_fields import as_number, as_weight_map
from comfyui_image_scorer.core.filesystem.paths import text_data_file
from comfyui_image_scorer.core.io.serialization import extract_prompt_tags, load_single_jsonl
from comfyui_image_scorer.core.observability.logger import (
    ModuleLogger,
    configure_package_logging,
    get_logger,
)
from comfyui_image_scorer.domain.analysis.trueskill import (
    public_score_from_rating,
    replay_ratings,
    trueskill_score_from_rating,
)
from comfyui_image_scorer.domain.training.combination_search import (
    Element,
    indicator_matrix,
    nested_cross_validation,
    permutation_null,
)
from comfyui_image_scorer.domain.training.statistics import max_statistic_bootstrap
from comfyui_image_scorer.domain.vectors.terms import extract_terms
from comfyui_image_scorer.infrastructure.loading.training_loader import training_loader
from comfyui_image_scorer.infrastructure.ml_models.plot import PlotManager
from comfyui_image_scorer.infrastructure.persistence.path_handler import find_image_path

# 20 = INFO. __name__ is __main__ in a notebook, and configure_package_logging
# sets that logger's level, so anything higher drops every info record.
configure_package_logging(20)
logger: ModuleLogger = get_logger(__name__)

## 1. The Toggle, and What 0.1 Means

`SCORE_FORM` switches the target between the two score forms the module
produces:

- `"public"` — the public score, a 0 to 1 bounded value the ranking model uses.
- `"trueskill"` — the raw skill value, `mu - 3 * sigma`, which is unbounded.

`MOVE` is the move worth confirming, stated in **public score** units. The
threshold drawn on the chart is that same move converted into the selected
target's units through the ratio of the two targets' standard deviations, so the
toggle compares like with like. Without that conversion a fixed 0.1 would be a
third of a standard deviation on the bounded public score and a hundredth of one
on the unbounded trueskill value, and every trueskill bar would clear it
trivially.

Conversion by standard deviation is an approximation, because the public score is
a bounded transform of the skill value rather than a linear one, so the two
targets do not move in exact proportion. It is close enough to keep the
comparison honest, and the logged ratio is printed so the size of the
approximation is visible.

In [ ]:
SCORE_FORM: str = "public"

# The move worth confirming, in public score units. The chart's threshold is
# this value converted into the selected target's own units below.
MOVE: float = 0.1

# A condition needs this many images on both sides of its comparison, so a term
# seen a handful of times cannot appear as a finding, and a condition carried by
# nearly every image has no comparison group to measure against at all.
MIN_SUPPORT: int = 100

# Quantile bins per numeric parameter. The cut points come from the data.
NUMERIC_BINS: int = 4

# Beam width and depth for the combination search.
MAX_ELEMENTS: int = 5
BEAM_WIDTH: int = 12

# Every fold selects its own winner from the rest and is scored on the fold it
# never saw, so what comes out is what the procedure yields on unseen data rather
# than the best of one lucky split.
FOLD_COUNT: int = 5

# The null: the same search on shuffled scores. This is what answers "would this
# have found something anyway", which no interval and no held-out half answers.
# Each round costs one search per fold, so this is the notebook's long pole.
# 50 is the practical floor: a permutation p-value cannot resolve below 1/(rounds
# + 1), so fewer than about 20 rounds cannot demonstrate a 0.05 threshold at all.
PERMUTATION_ROUNDS: int = 50

# Resamples behind the family-wise threshold on the single conditions.
BOOTSTRAP_ITERATIONS: int = 2000

SEED: int = 20251015

# Conditions carried onto the chart. The chart is meant to be read at a glance,
# so it shows the strongest few rather than every candidate.
TOP_CHART: int = 24

# Columns in the example image grid, per end of the score range. Only affects
# how many pictures are shown, never a measured number.
EXAMPLE_GRID: int = 4

SCORE_FORMS = {
    "public": public_score_from_rating,
    "trueskill": trueskill_score_from_rating,
}
if SCORE_FORM not in SCORE_FORMS:
    raise ValueError(f"SCORE_FORM must be one of {sorted(SCORE_FORMS)}")
score_of = SCORE_FORMS[SCORE_FORM]

replay_started = time.perf_counter()
comparison_rows = training_loader.load_comparison_rows()
replayed = replay_ratings(comparison_rows, order="default")
ratings = {name: rating for name, (rating, _count) in replayed.items()}

# Both spreads are needed to convert the move between the two targets, so both
# are measured on the same replay rather than one of them being assumed.
spreads = {
    form: float(np.std([score_of(rating) for rating in ratings.values()], ddof=1))
    for form, function in SCORE_FORMS.items()
}
target_sd = spreads[SCORE_FORM]
THRESHOLD: float = MOVE * target_sd / spreads["public"]

scores = {name: score_of(rating) for name, rating in ratings.items()}
target_values = np.array(list(scores.values()), dtype=np.float64)

logger.info("replayed %d comparisons over %d images in %.0fs",
            len(comparison_rows), len(scores), time.perf_counter() - replay_started)
logger.info("target '%s': min %.4f, max %.4f, mean %.4f, sd %.4f",
            SCORE_FORM, target_values.min(), target_values.max(),
            target_values.mean(), target_sd)
logger.info("the move is %.2f public score, which is %.4f in '%s' units",
            MOVE, THRESHOLD, SCORE_FORM)
logger.info("that is %.0f%% of the target's standard deviation",
            100.0 * THRESHOLD / target_sd)

### What Each Variable Does

Every knob lives in the block above and nothing below redefines it, so changing
one and re-running changes the whole analysis consistently. The table says what
each one actually buys, because several of them trade against each other.

| Variable | What it controls | Unit | What moving it does |
|---|---|---|---|
| `SCORE_FORM` | which score form is the target | `"public"` / `"trueskill"` | changes the numbers *and* what the threshold means; the method is identical either way |
| `MOVE` | the effect worth confirming, stated in public-score units | score points | moves the bar. It is an input to the question, never something the analysis adjusts |
| `MIN_SUPPORT` | images required on **each** side of a comparison | images | the rarity floor for every condition, every combination, and every fold measurement. Raising it drops weak findings and the rare ones together |
| `NUMERIC_BINS` | quantile bins per numeric parameter | bins | finer or coarser labels for cfg, steps, clip_skip and the size fields. Fewer bins means broader, more common conditions |
| `MAX_ELEMENTS` | deepest combination searched | conditions | more interaction can be found, and each extra condition is rarer and noisier |
| `BEAM_WIDTH` | partial combinations kept alive per step | combinations | `1` reproduces the greedy chain exactly. More beams return alternatives instead of one path, at roughly quadratic cost |
| `FOLD_COUNT` | folds in cross-validation | folds | more folds gives each selection more training data and a tighter mean, at a linear cost in searches |
| `PERMUTATION_ROUNDS` | shuffled reruns of the entire search | rounds | the resolution of the null's tail, and the notebook's long pole. A permutation p-value cannot resolve below `1 / (rounds + 1)`, so below about 20 rounds a 0.05 threshold becomes unprovable |
| `BOOTSTRAP_ITERATIONS` | resamples behind the family-wise threshold | resamples | how stable that threshold is, not how significant anything is |
| `SEED` | fixes the split, the fold assignment and every shuffle | integer | same seed reproduces the same numbers exactly. Changing it is the honest way to see how much of the result is split luck |
| `TOP_CHART` | conditions carried onto the chart | rows | display only. It cannot change any number, because the corrections run over the whole screen regardless |
| `EXAMPLE_GRID` | columns in the example image grid, per end of the score range | images | display only, and only affects how many pictures are shown |

Three of these are worth reading twice, because they are the ones that decide
whether a result is real:

- **`MIN_SUPPORT` does double duty under cross-validation.** It is the floor for
  the *measurement*, so the search is internally told to hold
  `MIN_SUPPORT * FOLD_COUNT` on the rows it can see. Without that scaling the
  search returns a winner too rare to measure on the fold meant to score it.
- **`BEAM_WIDTH` and `MAX_ELEMENTS` both cost support.** Every extra condition
  multiplies rarity, which is why the beam is what turns one chain into a set of
  alternatives rather than simply searching harder.
- **`SEED` does not make a result more reliable.** It makes it reproducible. The
  spread across seeds is the uncertainty; a single seed quietly hides it.

The quantities computed from them are named for what they measure, not for the
step that produced them:

| Name | What it holds |
|---|---|
| `spreads`, `target_sd` | standard deviation of each score form, measured on the same replay so the two are comparable |
| `THRESHOLD` | `MOVE` converted into the active target's units via `spreads` |
| `comparable` | mask of conditions carrying images on both sides; the rest have no effect to measure |
| `family` | observed effects, family-wise p-values, and the critical value, for the whole screen |
| `single_rows` | one row per condition, ordered by effect size |
| `cross_validated` | the per-fold selections behind the combination figure |
| `null_lifts` | mean out-of-fold lift per shuffled round: what the search invents |
| `observed_mean` | mean out-of-fold lift on the real scores, which is what has to beat `null_lifts` |

## 2. The Conditions

One pass over the text sidecar collects, per image, the prompt terms and the
generation parameters. Numeric parameters become quantile bins labelled with
their own edges, so the cut points are data rather than a choice made here.

The candidate set is every condition reaching `MIN_SUPPORT`. Conditions below
the floor are counted and dropped, so rarity cannot masquerade as influence.

In [ ]:
CATEGORICAL_FAMILIES = ("model", "sampler", "scheduler", "lora")
NUMERIC_FAMILIES = (
    "cfg",
    "steps",
    "clip_skip",
    "final_width",
    "final_height",
    "final_aspect_ratio",
)
PROMPT_FAMILIES = ("positive_prompt", "negative_prompt")


def read_conditions(scored: dict[str, float]) -> dict[str, list[Element]]:
    """Collect the per-image conditions for every scored image.

    Returns a mapping of filename to the conditions that image carries, which is
    the only shape the search needs: a condition is present on an image or it is
    not, and the search works in terms of presence.
    """
    prompts: dict[str, dict[str, str]] = defaultdict(dict)
    categorical: dict[str, dict[str, str]] = defaultdict(dict)
    numeric: dict[str, dict[str, float]] = defaultdict(dict)

    for row in load_single_jsonl(text_data_file):
        filename, payload = next(iter(row.items()))
        if filename not in scored:
            continue

        positive = extract_prompt_tags(payload)
        if positive:
            prompts[filename]["positive_prompt"] = positive
        negative = payload.get("negative_prompt")
        if isinstance(negative, str) and negative:
            prompts[filename]["negative_prompt"] = negative

        for family in CATEGORICAL_FAMILIES:
            weights = as_weight_map(payload.get(family))
            if weights:
                name, _weight = max(weights.items(), key=lambda item: item[1])
                if name:
                    categorical[filename][family] = name

        for family in NUMERIC_FAMILIES:
            value = as_number(payload.get(family))
            if value is not None:
                numeric[filename][family] = value

    elements: dict[str, list[Element]] = {}
    for filename, family_prompts in prompts.items():
        carried: list[Element] = []
        for family, text in family_prompts.items():
            terms = [term for term, _weight, _index in extract_terms(text).terms]
            carried.extend(Element(family, term) for term in terms)
        for family, name in categorical[filename].items():
            carried.append(Element(family, name))
        elements[filename] = carried

    # The bin edges are taken from the values actually present, so a parameter
    # with a narrow range does not get bins nobody lands in.
    edges_by_family: dict[str, list[float]] = {}
    for family in NUMERIC_FAMILIES:
        column = np.array(
            [numeric[name][family] for name in elements if family in numeric[name]],
            dtype=np.float64,
        )
        if column.size < MIN_SUPPORT:
            continue
        edges_by_family[family] = [
            float(edge)
            for edge in np.unique(np.quantile(column, np.linspace(0.0, 1.0, NUMERIC_BINS + 1)))
        ]

    def bin_label(family: str, value: float) -> str:
        edges = edges_by_family[family]
        for index in range(len(edges) - 1):
            if edges[index] <= value <= edges[index + 1]:
                low, high = f"{edges[index]:.2f}", f"{edges[index + 1]:.2f}"
                return low if index + 2 == len(edges) else f"{low}-{high}"
        return f"{value:.2f}"

    for filename, carried in elements.items():
        for family in NUMERIC_FAMILIES:
            value = numeric[filename].get(family)
            if value is not None and family in edges_by_family:
                carried.append(Element(family, bin_label(family, value)))

    return elements


read_started = time.perf_counter()
elements_by_image = read_conditions(scores)

condition_counts: Counter[Element] = Counter()
for carried in elements_by_image.values():
    condition_counts.update(carried)
candidates = sorted(
    element for element, count in condition_counts.items() if count >= MIN_SUPPORT
)
by_family = Counter(element.family for element in candidates)
logger.info(
    "%d conditions above the floor of %d, %d dropped for rarity, in %.0fs",
    len(candidates),
    MIN_SUPPORT,
    len(condition_counts) - len(candidates),
    time.perf_counter() - read_started,
)
logger.info("per family: %s", ", ".join(f"{name}={count}" for name, count in by_family.most_common()))

## 3. Three Corrections, Because One Is Not Enough

A number with an interval still is not a finding, for three separate reasons,
and each has its own correction here.

**Multiplicity.** Thousands of conditions are screened and only the extremes get
reported. Correcting the reported handful for multiplicity corrects the wrong
family, because that handful is already the tail of the screen. The
family-wise max-T bootstrap instead asks how large the *largest* effect gets
across all conditions under the same resampling, and clears each condition
against that. Its threshold widens automatically when conditions overlap, so a
large effect shared by many of them is not counted many times.

**Selection.** The search returns the best of a thousand candidates on the images
it searched, so that lift is a maximum and sits higher than any single
combination deserves. Every fold therefore selects its own winner from the other
folds and is scored on the fold it never saw, and the additive baseline is fitted
on the training folds and merely applied to the held-out fold. What comes out is
what the procedure yields on unseen data.

**The bias that survives all of that.** Even a perfectly honest held-out
measurement cannot say whether the procedure would have found something anyway.
That needs the null: the same folds, the same search, the same everything, on
shuffled scores where no association exists. The bar is wherever the search
reaches when it is searching nothing.

Each of the three is reported separately, because a result that survives one and
not another is a different kind of wrong.

In [ ]:
# The three shapes the search and the corrections both work from: which
# conditions each image carries, the score of each image, and the presence matrix
# those two produce together.
image_names = list(elements_by_image)
records = [elements_by_image[name] for name in image_names]
record_scores = np.array([scores[name] for name in image_names], dtype=np.float64)
matrix = indicator_matrix(records, candidates)
logger.info("indicator matrix %s over %d images", matrix.shape, len(records))

# --- single conditions, corrected across the whole screen ---------------------
# The family is every comparable condition, not the handful that reach the chart.
# Correcting the reported few would correct for the wrong family, because those
# few are already the extreme tail of the screen.
with_count = matrix.sum(axis=0)
without_count = len(record_scores) - with_count
comparable = (with_count >= MIN_SUPPORT) & (without_count >= MIN_SUPPORT)
logger.info(
    "%d of %d conditions have a comparison group on both sides; %d are carried by "
    "nearly every image and have no effect to measure",
    int(comparable.sum()),
    len(candidates),
    int((~comparable).sum()),
)

with tqdm(
    total=BOOTSTRAP_ITERATIONS,
    desc="family-wise threshold",
    unit="resample",
) as family_bar:
    family = max_statistic_bootstrap(
        matrix[:, comparable],
        record_scores,
        iterations=BOOTSTRAP_ITERATIONS,
        seed=SEED,
        progress=lambda done: family_bar.update(done - family_bar.n),
    )
logger.info(
    "family-wise threshold over %d conditions: the largest effect expected from "
    "noise alone at 95%% is %.4f",
    int(comparable.sum()),
    family.critical_value,
)

# One row per condition that had images on both sides, ordered by effect size so
# the strongest are first. `n` is how many images carried the condition, which is
# the evidence behind the effect.
single_rows = pd.DataFrame(
    {
        "label": [str(candidates[index]) for index in np.flatnonzero(comparable)],
        "kind": "single",
        "influence": family.observed,
        "family_p_value": family.p_values,
        "n": with_count[comparable].astype(int),
    }
).sort_values("influence", key=abs, ascending=False)
logger.info(
    "%d of %d conditions clear the family-wise threshold",
    int((single_rows["family_p_value"] <= 0.05).sum()),
    len(single_rows),
)

## 4. What The Procedure Yields On Unseen Data

Cross-validation replaces the single held-back half. Each fold picks its own
winner and is scored on the fold it never saw, so no fold's number is inflated
by having chosen its combination on those same images.

The figure carried forward is the **mean across folds**, not the best one. The
best of five lifts is a maximum, and quoting it would put back the exact
selection bias the folds exist to remove. The per-fold numbers are printed anyway
so the spread and the worst case are visible, because the worst fold is what it
would cost to act on the finding and have it be wrong.

How often the folds **agreed** separates a result the search converges on from one
it keeps redrawing.

In [ ]:
with tqdm(total=FOLD_COUNT, desc="cross-validation folds", unit="fold") as fold_bar:
    cross_validated = nested_cross_validation(
        records,
        record_scores,
        candidates,
        fold_count=FOLD_COUNT,
        beam_width=BEAM_WIDTH,
        max_elements=MAX_ELEMENTS,
        min_support=MIN_SUPPORT,
        seed=SEED,
        progress=lambda done: fold_bar.update(done - fold_bar.n),
    )

# One row per fold: which combination that fold chose on its own, how many
# held-out images matched it, and the lift there. Printed rather than plotted so
# the spread and the worst fold stay visible next to the mean.
fold_rows = pd.DataFrame(
    [
        {
            "fold": selection.fold,
            "label": selection.label,
            "depth": len(selection.columns),
            "n": selection.support,
            "observed": selection.observed_score,
            "predicted": selection.predicted_score,
            "lift": selection.lift,
        }
        for selection in cross_validated.selections
    ]
)
display(fold_rows.round(4))
logger.info(
    "out-of-fold lift: mean %+.4f, best %+.4f, worst %+.4f over %d folds "
    "(%d unmeasurable)",
    cross_validated.mean_lift,
    cross_validated.best_lift,
    min(cross_validated.lifts, default=0.0),
    len(cross_validated.selections),
    cross_validated.unmeasurable_folds,
)
logger.info(
    "folds chose %d distinct combination(s); most repeated: %s",
    len(cross_validated.repetitions),
    cross_validated.most_repeated_label or "none",
)

## 5. What The Search Invents From Nothing

The same folds, the same search, the same support rules, run on scores shuffled
across images. Shuffling destroys every association the design implies while
leaving the design, the sample sizes and the score distribution untouched, so
every lift this produces is exactly what the procedure manufactures from noise.

This is the bar the previous section has to clear. It is deliberately the most
expensive cell in the notebook: each round runs a full search per fold.

In [ ]:
with tqdm(
    total=PERMUTATION_ROUNDS,
    desc="permutation null",
    unit="round",
) as null_bar:
    null_lifts = permutation_null(
        records,
        record_scores,
        candidates,
        rounds=PERMUTATION_ROUNDS,
        fold_count=FOLD_COUNT,
        beam_width=BEAM_WIDTH,
        max_elements=MAX_ELEMENTS,
        min_support=MIN_SUPPORT,
        seed=SEED,
        progress=lambda done: null_bar.update(done - null_bar.n),
    )

null_p95 = float(np.quantile(null_lifts, 0.95))
null_max = float(null_lifts.max())
# The mean, not the best fold. The best of FOLD_COUNT lifts is a maximum, and
# quoting it would put the very selection bias this notebook exists to remove
# straight back into the headline number. The null is a mean for the same reason,
# so the two are on one scale and can be compared directly.
observed_mean = cross_validated.mean_lift
observed_best = cross_validated.best_lift
# The share of shuffled rounds the observed value sits above, which is the
# permutation test's p-value read straight off the rounds that were run.
beats_null = float((null_lifts < observed_mean).mean())
# The p-value is the other direction: how often a shuffled run reached at least
# as high. Beating every round means beating none of them, so the p-value is the
# complement. Testing the share-beaten against 0.05 would invert the whole test
# and mark the most significant result as the least.
null_p_value = float((null_lifts >= observed_mean).mean())

logger.info(
    "null over %d rounds: p50 %+.4f, p95 %+.4f, largest %+.4f",
    PERMUTATION_ROUNDS,
    float(np.quantile(null_lifts, 0.5)),
    null_p95,
    null_max,
)
logger.info(
    "observed mean out-of-fold lift is %+.4f, which is above %d of the %d "
    "shuffled rounds",
    observed_mean,
    int(round(beats_null * PERMUTATION_ROUNDS)),
    PERMUTATION_ROUNDS,
)
logger.info(
    "permutation p-value %.4f (a finite run cannot resolve below %.4f)",
    null_p_value,
    1.0 / (PERMUTATION_ROUNDS + 1),
)
logger.info(
    "per fold the lifts run %+.4f to %+.4f, mean %+.4f",
    min(cross_validated.lifts, default=0.0),
    observed_best,
    observed_mean,
)
verdict = (
    "beyond what this search invents from noise"
    if observed_mean > null_p95
    else "within what this search produces on shuffled scores, so not separable from noise"
)
logger.info("verdict: %s", verdict)

In [ ]:
# --- one table, all three corrections ----------------------------------------
# The search result is its own row so it cannot be lost among the single
# conditions. It carries the same two verdicts as they do, but arrived at a
# different way: clearing the screen means beating the permutation null rather
# than clearing the family-wise threshold, because a combination is chosen by a
# search rather than screened as one of thousands.
SEARCH_LABEL = f"\u2b50 SEARCH RESULT: best combination, mean of {FOLD_COUNT} out-of-fold folds"

chart = pd.concat(
    [
        single_rows.head(TOP_CHART).assign(
            clears_threshold=lambda frame: frame["influence"].abs() >= THRESHOLD,
            clears_family=lambda frame: frame["family_p_value"] <= 0.05,
        ),
        pd.DataFrame(
            [
                {
                    "label": SEARCH_LABEL,
                    "kind": "combination",
                    "influence": observed_mean,
                    "family_p_value": null_p_value,
                    "n": int(fold_rows["n"].max()) if not fold_rows.empty else 0,
                    "clears_threshold": abs(observed_mean) >= THRESHOLD,
                    "clears_family": null_p_value <= 0.05,
                }
            ]
        ),
    ],
    ignore_index=True,
).sort_values("influence", key=abs, ascending=False)

display(chart.round(4).reset_index(drop=True))
logger.info(
    "single conditions at or past %.2f: %d of %d screened",
    THRESHOLD,
    int(single_rows["influence"].abs().ge(THRESHOLD).sum()),
    len(single_rows),
)
logger.info(
    "of those, also clearing the family-wise threshold: %d",
    int(
        (
            single_rows["influence"].abs().ge(THRESHOLD)
            & (single_rows["family_p_value"] <= 0.05)
        ).sum()
    ),
)
logger.info(
    "the search's mean out-of-fold lift is %+.4f, %s %.2f",
    observed_mean,
    "at or past" if abs(observed_mean) >= THRESHOLD else "short of",
    THRESHOLD,
)

## 6. The Chart

One figure, both measures, with both bars that decide it drawn on it.

- The **red dashed lines** are the move being tested for, in the selected
  target's units.
- The **grey line** is the family-wise threshold: the largest effect the whole
  screen produces from noise alone. A bar left of it has not cleared the screen,
  however large it looks.
- The **black dotted line** is where the search lands on shuffled scores. A
  combination bar left of it is inside what this procedure invents when there is
  nothing to find.
- **Green** clears both its threshold and the requested move. **Amber** clears
  the move but not the screen, which is the large-but-not-established case.
  **Grey** does not clear the move at all.

Single conditions are effects against images without them. The combination bar
is the **mean** lift the search achieved out of fold, with the whiskers spanning
the folds individually: the bar is what the procedure delivers, and the whiskers
show what a single fold gave, including the worst one. A lift is smaller than a
raw gap by construction, because it is what the conditions do to *each other*
once their individual effects are accounted for.

In [ ]:
chart = chart.sort_values("influence", key=abs).reset_index(drop=True)


def _colour(row) -> str:
    """Three states, and the middle one matters most.

    A bar that clears one test and not the other is not a weak bar. It cleared
    something real and missed the bar that was asked about, which is a different
    answer from having cleared nothing. Painting both as the same grey made a
    confirmed result indistinguishable from noise.
    """
    if row.clears_threshold and row.clears_family:
        return "tab:green"
    if row.clears_threshold or row.clears_family:
        return "tab:orange"
    return "tab:grey"


colours = [_colour(row) for row in chart.itertuples()]

figure, axis = plt.subplots(figsize=(12, max(6.0, 0.32 * len(chart))))
is_combination = chart["kind"].eq("combination")
# The search result is hatched and outlined even when it clears nothing, so it
# can never be mistaken for a condition that was measured and found wanting.
axis.barh(
    range(len(chart)),
    chart["influence"],
    color=colours,
    edgecolor="black",
    linewidth=0.4,
    alpha=0.85,
    hatch=["///" if flag else "" for flag in is_combination],
)
# Only the combination bar carries a spread, because only it is an average: a
# single condition's effect is one difference of means over all the images.
combination_row = int(chart["kind"].eq("combination").idxmax()) if is_combination.any() else None
if combination_row is not None and cross_validated.selections:
    axis.errorbar(
        observed_mean,
        combination_row,
        xerr=[
            [observed_mean - min(cross_validated.lifts, default=observed_mean)],
            [max(cross_validated.lifts, default=observed_mean) - observed_mean],
        ],
        fmt="none",
        ecolor="black",
        elinewidth=1.6,
        capsize=5,
    )
for position, value in enumerate(chart["influence"]):
    axis.text(
        value,
        position,
        f" {value:+.3f}",
        va="center",
        ha="left" if value >= 0 else "right",
        fontsize=8,
    )

for offset in (-THRESHOLD, THRESHOLD):
    axis.axvline(offset, color="tab:red", linestyle="--", linewidth=1.3)
axis.axvline(family.critical_value, color="grey", linestyle="-.", linewidth=1.3)
axis.axvline(null_p95, color="black", linestyle=":", linewidth=1.3)
axis.axvline(0.0, color="black", linewidth=1.0)
# A hard rule above the search result so it reads as its own section rather than
# as the smallest bar in a list of single conditions.
if combination_row is not None and combination_row > 0:
    axis.axhline(
        combination_row - 0.5, color="black", linewidth=1.8, linestyle="-", zorder=3
    )

axis.set_yticks(range(len(chart)))
axis.set_yticklabels(
    [
        f"{row.label[:74]}  (n={row.n:,})" if row.kind == "single"
        else f"{row.label[:74]}  (whiskers: {len(cross_validated.selections)} folds, worst {min(cross_validated.lifts, default=0.0):+.3f})"
        for row in chart.itertuples()
    ],
    fontsize=8,
)
axis.set_xlabel(f"effect on the '{SCORE_FORM}' target")
axis.set_title(
    f"What moves the {SCORE_FORM} score by {MOVE:.2f}? ({SCORE_FORM} units)\n"
    f"green: clears the move and the screen | orange: clears one of them | grey: clears neither\n"
    f"red: the {MOVE:.2f} move | grey dash-dot: largest effect from noise across the screen "
    f"({family.critical_value:.3f}) | black dotted: what the search invents on shuffled scores ({null_p95:.3f})"
)
axis.grid(axis="x", linestyle=":", alpha=0.4)
figure.tight_layout()
plt.show()

## 7. Which Kinds Of Conditions Appear

A combination of five prompt terms and a combination of five generation
parameters are different claims about how this dataset was made, so counting the
families across the folds shows which kinds of condition the search leaned on.

The count is taken over the fold selections rather than over everything the beam
produced. The beam returns dozens of combinations per fold, and reporting those
would be reporting the search's browsing, not its answer. A family that shows up
in only one of five folds is barely supported, and the count makes that visible
without needing a second chart.

A combination made entirely from one family is a claim about prompt wording or
about parameters, never about the two together, which is the kind of result worth
knowing about before acting on it.

In [ ]:
# The families each fold's chosen combination drew on, taken from the column
# indices rather than by parsing the label back apart.
fold_families = pd.DataFrame(
    [
        {
            "fold": selection.fold,
            "depth": len(selection.columns),
            "families": sorted({candidates[column].family for column in selection.columns}),
            "lift": selection.lift,
        }
        for selection in cross_validated.selections
    ]
)

if fold_families.empty:
    logger.info("no fold produced a measurable combination, so there is nothing to break down")
else:
    family_counts: Counter[str] = Counter()
    for families in fold_families["families"]:
        family_counts.update(families)
    logger.info(
        "families drawn on across %d fold selection(s): %s",
        len(fold_families),
        ", ".join(f"{name}={count}" for name, count in family_counts.most_common()),
    )
    for row in fold_families.itertuples():
        logger.info(
            "  fold %d, %d condition(s), lift %+.4f: %s",
            row.fold,
            row.depth,
            row.lift,
            ", ".join(row.families),
        )
    logger.info(
        "combinations drawn from a single family: %d of %d",
        int(fold_families["families"].map(len).eq(1).sum()),
        len(fold_families),
    )

## 8. Examples

The combination the folds agreed on most, shown as the actual images that carry
every one of its conditions. This is a check on the finding rather than more
analysis: if the conditions describe what is visibly in the pictures the result
is readable, and if they plainly do not, then the search found structure in the
metadata that the image does not carry, which is a different and less useful
claim than it looks.

The rows picked are the highest and lowest scoring matches, so both ends of the
range are visible rather than only the flattering one. The combination the folds
agree on is used rather than the single best fold, because the best fold is a
maximum and this is not the place to quote one.

In [ ]:
def _resolve(filename: str) -> str | None:
    path = find_image_path(filename)
    return str(path) if path is not None else None


example_label = cross_validated.most_repeated_label
example_columns = next(
    (
        selection.columns
        for selection in cross_validated.selections
        if selection.label == example_label
    ),
    (),
)
required = {candidates[column] for column in example_columns}

if not required:
    logger.info("no fold produced a repeated combination, so there is nothing to illustrate")
else:
    matching = sorted(
        name for name, elements in elements_by_image.items() if required.issubset(set(elements))
    )
    # Highest and lowest scoring matches, so the range is visible and not just
    # the end of it that flatters the finding.
    picked = matching[:EXAMPLE_GRID] + matching[-EXAMPLE_GRID:]
    logger.info(
        "%d of %d images carry every condition of the combination the folds agreed on (%d conditions)",
        len(matching),
        len(elements_by_image),
        len(required),
    )
    logger.info("conditions: %s", example_label)
    PlotManager.plot_image_grid(
        filenames=picked,
        titles=[f"{scores[name]:.3f}" for name in picked],
        path_for=_resolve,
        columns=EXAMPLE_GRID,
        title=f"Images carrying every condition of:\n{example_label}",
    )